IDX Exchange DS47 Code: Joe Hiller
Initial step: Create train-test split. For first split: read in six months of data from 06-2025 to 11-2025. Use 12-2025 as test set. For second split: read in six months of data from 07-2025 to 12-2025. Use 1-2026 as test set.



For both train and test, ensure that property type is residential and propertySubType is single family residence. 

According to Wikipedia, the latitude and longitude ranges for California are:
Latitude	32°32′ N to 42° N
Longitude	114°8′ W to 124°26′ W
Angle units are defined in decimal parts by dividing by 60. e.g. 32' = 32/60.

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error as mape
import numpy as np

LAT_MIN = 32.0 + 32.0/60
LAT_MAX = 42.0
LON_MIN = -124.0 + 26.0/60
LON_MAX = -114.0 + 8.0/60


# reads csv files into dataframes
df0 = pd.read_csv("CRMLSSold202506.csv")
df1 = pd.read_csv("CRMLSSold202507.csv")
df2 = pd.read_csv("CRMLSSold202508.csv")
df3 = pd.read_csv("CRMLSSold202509.csv")
df4 = pd.read_csv("CRMLSSold202510.csv")
df5 = pd.read_csv("CRMLSSold202511.csv")
df6 = pd.read_csv("CRMLSSold202512.csv")
df7 = pd.read_csv("CRMLSSold202601.csv")
# Creates train test split ending in december
df_train_dec = pd.concat([df0, df1, df2, df3, df4, df5], axis = 0)
df_test_dec = df6
# Filters the data
df_train_dec = df_train_dec[(df_train_dec["PropertyType"] == "Residential") & (df_train_dec["PropertySubType"] == "SingleFamilyResidence")]
df_test_dec = df_test_dec[(df_test_dec["PropertyType"] == "Residential") & (df_test_dec["PropertySubType"] == "SingleFamilyResidence")]

# filters out invalid data
df_train_dec = df_train_dec[(df_train_dec["ClosePrice"] > 0) & (df_train_dec["LivingArea"] > 0)]
df_train_dec = df_train_dec[df_train_dec["StateOrProvince"] == "CA"] 


# filters out invalid test data
df_test_dec = df_test_dec[(df_test_dec["ClosePrice"] > 0) & (df_test_dec["LivingArea"] > 0)]
df_test_dec = df_test_dec[df_test_dec["StateOrProvince"] == "CA"]
# df_test_dec = df_test_dec.dropna(subset = ["Latitude", "Longitude"])

#filter to only properties in california

df_train_dec.loc[
    ~df_train_dec["Latitude"].between(LAT_MIN, LAT_MAX),
    "Latitude"
] = None

df_train_dec.loc[~df_train_dec["Longitude"].between(LON_MIN, LON_MAX), "Longitude"] = None


df_train_dec = df_train_dec.dropna(subset = ["Latitude", "Longitude"])

df_test_dec.loc[
    ~df_test_dec["Latitude"].between(LAT_MIN, LAT_MAX), 
    "Latitude"
] = None

df_test_dec.loc[
    ~df_test_dec["Longitude"].between(LON_MIN, LON_MAX),
    "Longitude"
] = None

df_test_dec = df_test_dec.dropna(subset = ["Latitude", "Longitude"])

#removes duplicates from train and test data

df_train_dec['CloseDate'] = pd.to_datetime(df_train_dec['CloseDate'])

df_train_dec = (
    df_train_dec.sort_values('CloseDate', ascending=False)
      .drop_duplicates(subset='ListingKey', keep='first')
)

df_test_dec['CloseDate'] = pd.to_datetime(df_test_dec['CloseDate'])
df_test_dec = (
    df_test_dec.sort_values('CloseDate', ascending=False)
      .drop_duplicates(subset="ListingKey", keep='first')
)




C:\Users\jehil\AppData\Local\Temp\ipykernel_17756\1688373431.py:12: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df0 = pd.read_csv("CRMLSSold202506.csv")
C:\Users\jehil\AppData\Local\Temp\ipykernel_17756\1688373431.py:19: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df7 = pd.read_csv("CRMLSSold202601.csv")


I created the train and test sets for the data period ending in January 2026. I applied the same data preprocessing techniques to this train and test set for consistency. 

In [2]:
df_train_jan = pd.concat([df1, df2, df3, df4, df5, df6], axis = 0)
df_test_jan = df7

df_train_jan = df_train_jan[(df_train_jan["PropertyType"] == "Residential") & (df_train_jan["PropertySubType"] == "SingleFamilyResidence")]
df_test_jan = df_test_jan[(df_test_jan["PropertyType"] == "Residential") & (df_test_jan["PropertySubType"] == "SingleFamilyResidence")]

# filters out invalid data
df_train_jan = df_train_jan[(df_train_jan["ClosePrice"] > 0) & (df_train_jan["LivingArea"] > 0)]
df_train_jan = df_train_jan[df_train_jan["StateOrProvince"] == "CA"] 


# filters out invalid test data
df_test_jan = df_test_jan[(df_test_jan["ClosePrice"] > 0) & (df_test_jan["LivingArea"] > 0)]
df_test_jan = df_test_jan[df_test_jan["StateOrProvince"] == "CA"]


#filter to only properties in california

df_train_jan.loc[
    ~df_train_jan["Latitude"].between(LAT_MIN, LAT_MAX),
    "Latitude"
] = None

df_train_jan.loc[~df_train_jan["Longitude"].between(LON_MIN, LON_MAX), "Longitude"] = None


df_train_jan = df_train_jan.dropna(subset = ["Latitude", "Longitude"])

df_test_jan.loc[
    ~df_test_jan["Latitude"].between(LAT_MIN, LAT_MAX), 
    "Latitude"
] = None

df_test_jan.loc[
    ~df_test_jan["Longitude"].between(LON_MIN, LON_MAX),
    "Longitude"
] = None

df_test_jan = df_test_jan.dropna(subset = ["Latitude", "Longitude"])

df_train_jan['CloseDate'] = pd.to_datetime(df_train_jan['CloseDate'])

df_train_jan = (
    df_train_jan.sort_values('CloseDate', ascending=False)
      .drop_duplicates(subset='ListingKey', keep='first')
)

df_test_jan['CloseDate'] = pd.to_datetime(df_test_jan["CloseDate"])

df_test_jan = (
    df_test_jan.sort_values('CloseDate', ascending=False)
      .drop_duplicates(subset='ListingKey', keep='first')
)








I know calculate the missing percentage for each variable. If the missing percentage is above 15% I don't include it in the model, as 15% of the data should not be imputed.

In [3]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


missing_dec = (df_train_dec.isna().sum() / len(df_train_dec)) * 100
missing_pcts = missing_dec.sort_values(ascending = False)
print(missing_pcts)
pd.reset_option("display.max_columns")
pd.reset_option("display.max_rows")

MiddleOrJuniorSchoolDistrict    100.000000
FireplacesTotal                 100.000000
AboveGradeFinishedArea          100.000000
TaxAnnualAmount                 100.000000
TaxYear                         100.000000
ElementarySchoolDistrict        100.000000
CoveredSpaces                   100.000000
BusinessType                    100.000000
WaterfrontYN                     99.960460
BelowGradeFinishedArea           99.283894
BasementYN                       97.541224
BuilderName                      95.426588
LotSizeDimensions                93.582872
BuildingAreaTotal                93.298773
CoBuyerAgentFirstName            90.673052
ElementarySchool                 86.831854
MiddleOrJuniorSchool             86.612190
HighSchool                       82.675805
CoListAgentFirstName             76.446124
CoListAgentLastName              76.374367
AssociationFeeFrequency          75.047594
CoListOfficeName                 73.354714
SubdivisionName                  64.541780
MainLevelBe

In [4]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


missing_jan = (df_train_jan.isna().sum() / len(df_train_jan)) * 100
missing_jan_percents = missing_jan.sort_values(ascending = False)
print(missing_jan_percents)
pd.reset_option("display.max_columns")
pd.reset_option("display.max_rows")

MiddleOrJuniorSchoolDistrict    100.000000
FireplacesTotal                 100.000000
AboveGradeFinishedArea          100.000000
TaxAnnualAmount                 100.000000
TaxYear                         100.000000
ElementarySchoolDistrict        100.000000
CoveredSpaces                   100.000000
BusinessType                    100.000000
WaterfrontYN                     99.959718
BelowGradeFinishedArea           99.271935
BasementYN                       97.617378
BuilderName                      95.537619
LotSizeDimensions                93.590643
BuildingAreaTotal                93.290763
CoBuyerAgentFirstName            90.802214
ElementarySchool                 87.041043
MiddleOrJuniorSchool             86.882898
HighSchool                       82.914348
CoListAgentFirstName             76.588539
CoListAgentLastName              76.521402
AssociationFeeFrequency          75.092127
CoListOfficeName                 73.446522
SubdivisionName                  64.808212
MainLevelBe

I found the intersection of features where both December and January endings appear. I removed ListPrice, OriginalListPrice, PurchaseContractDate, and CloseDate as it has to do with the time of sale.

In [5]:

cols = ["ClosePrice", "AttachedGarageYN", "Stories", "ViewYN", "PoolPrivateYN", "NewConstructionYN", "Levels", "BuyerOfficeAOR", "GarageSpaces", "LotSizeSquareFeet", "LotSizeAcres", "LotSizeArea", "BuyerOfficeName", "ListAgentFirstName", "BuyerAgentFirstName", "ListAgentEmail", "BuyerAgentMlsId", "StreetNumberNumeric", "UnparsedAddress", "FireplaceYN", "YearBuilt", "BuyerAgentLastName", "City", "ListAgentFullName", "BuyerAgentAOR", "BathroomsTotalInteger", "ListAgentAOR", "ListAgentLastName", "PostalCode", "ParkingTotal", "PropertyType", "ListingKeyNumeric", "ListOfficeName", "CountyOrParish", "MlsStatus", "DaysOnMarket", "ListingKey", "LivingArea", "PropertySubType", "ContractStatusChangeDate", "ListingId", "StateOrProvince", "ListingContractDate", "Latitude", "Longitude", "BedroomsTotal"]                            

df_train_dec = df_train_dec[cols]

df_test_dec = df_test_dec[cols]
c_price = "ClosePrice"
test_low = df_test_dec[c_price].quantile(0.005)
test_high = df_test_dec[c_price].quantile(0.995)
df_test_dec = df_test_dec[(df_test_dec[c_price] >= test_low) & (df_test_dec[c_price] <= test_high)]
df_test_dec


,ClosePrice,AttachedGarageYN,Stories,ViewYN,PoolPrivateYN,NewConstructionYN,Levels,BuyerOfficeAOR,GarageSpaces,LotSizeSquareFeet,...,ListingKey,LivingArea,PropertySubType,ContractStatusChangeDate,ListingId,StateOrProvince,ListingContractDate,Latitude,Longitude,BedroomsTotal
0,1998000.0,True,1.0,NaN,False,False,One,ContraCosta,3.0,10080.0,...,1150041639,2045.0,SingleFamilyResidence,2025-12-31,41119782,CA,2025-12-31,37.871927,-122.029871,4.0
4072,1189000.0,True,1.0,True,False,False,One,OrangeCounty,2.0,4888.0,...,1146066190,1900.0,SingleFamilyResidence,2025-12-31,OC25254021,CA,2025-11-04,33.649317,-117.629635,2.0
13836,455000.0,True,1.0,NaN,True,False,One,CitrusValley,2.0,6534.0,...,1135662476,1687.0,SingleFamilyResidence,2025-12-31,IV25218752,CA,2025-09-17,33.957642,-116.999962,2.0
1437,5275000.0,NaN,2.0,False,False,False,Two,NaN,NaN,5301.0,...,1147260067,2880.0,SingleFamilyResidence,2025-12-31,25623811,CA,2025-12-01,33.992829,-118.461618,3.0
13686,900000.0,True,2.0,True,False,False,Two,PacificWest,2.0,7841.0,...,1135751174,2575.0,SingleFamilyResidence,2025-12-31,IG25220572,CA,2025-09-18,33.871405,-117.604177,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14307,915000.0,False,1.0,False,False,False,One,PacificWest,2.0,5882.0,...,1133924879,1318.0,SingleFamilyResidence,2025-12-01,IG25216123,CA,2025-09-12,33.975442,-118.032678,3.0
14348,740894.0,NaN,2.0,False,False,False,Two,Mrmls,2.0,0.0,...,1133869432,1691.0,SingleFamilyResidence,2025-12-01,SDC0001204SD,CA,2025-09-06,33.345481,-117.150766,3.0
14373,640000.0,False,1.0,False,False,False,One,Glendale,NaN,6326.0,...,1133803889,1188.0,SingleFamilyResidence,2025-12-01,GD25215487,CA,2025-09-12,33.927084,-118.331328,3.0
14375,1300000.0,False,1.0,True,False,False,One,SouthBay,2.0,7005.0,...,1133802220,1341.0,SingleFamilyResidence,2025-12-01,SB25197276,CA,2025-05-26,33.963929,-118.419502,3.0


I applied the same columns to the data ending in January and found the range of prices in that area

In [6]:



df_train_jan = df_train_jan[cols]

df_test_jan = df_test_jan[cols]
c_price = "ClosePrice"
test_low = df_test_jan[c_price].quantile(0.005)
test_high = df_test_jan[c_price].quantile(0.995)
df_test_jan = df_test_jan[(df_test_jan[c_price] >= test_low) & (df_test_jan[c_price] <= test_high)]
df_test_jan


,ClosePrice,AttachedGarageYN,Stories,ViewYN,PoolPrivateYN,NewConstructionYN,Levels,BuyerOfficeAOR,GarageSpaces,LotSizeSquareFeet,...,ListingKey,LivingArea,PropertySubType,ContractStatusChangeDate,ListingId,StateOrProvince,ListingContractDate,Latitude,Longitude,BedroomsTotal
13087,432000.0,True,1.0,False,False,False,One,Conejo,2.0,20500.0,...,1134739945,1140.0,SingleFamilyResidence,2026-01-31,SR25218548,CA,2025-09-16,34.655489,-118.205478,3.0
10231,840000.0,True,2.0,True,False,False,Two,RanchoSoutheast,2.0,5107.0,...,1144635049,2791.0,SingleFamilyResidence,2026-01-31,WS25246759,CA,2025-10-07,34.009959,-117.593657,5.0
2,2975000.0,True,1.0,NaN,False,False,One,NaN,0.0,238186.0,...,1151959385,1600.0,SingleFamilyResidence,2026-01-31,41122676,CA,2026-01-31,37.878010,-121.868460,2.0
12271,770000.0,False,1.0,False,False,False,One,Arcadia,2.0,6104.0,...,1137316585,1603.0,SingleFamilyResidence,2026-01-31,DW25228486,CA,2025-09-29,33.942085,-118.043587,3.0
1627,1350000.0,True,1.0,NaN,False,False,One,Mlslistings,2.0,6760.0,...,1150114839,1269.0,SingleFamilyResidence,2026-01-30,41119886,CA,2026-01-02,37.541337,-121.978632,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14741,350000.0,False,NaN,False,NaN,False,NaN,Mlslistings,2.0,6098.0,...,1120193042,1119.0,SingleFamilyResidence,2026-01-01,ML82016477,CA,2025-07-31,38.480602,-121.500243,3.0
12781,810000.0,NaN,1.0,False,False,False,One,Southland,2.0,5021.0,...,1136724065,1747.0,SingleFamilyResidence,2026-01-01,DW25222646,CA,2025-09-11,34.220024,-118.394489,3.0
15480,410000.0,NaN,1.0,True,True,False,One,SierraNorthValley,0.0,20473.0,...,1118267281,1352.0,SingleFamilyResidence,2026-01-01,SN25137783,CA,2025-06-19,39.799295,-121.895948,4.0
15617,710000.0,True,1.0,True,False,False,One,NorthSanLuisObispo,2.0,87120.0,...,1115109409,1873.0,SingleFamilyResidence,2026-01-01,PI25128639,CA,2025-05-28,35.634321,-120.547108,4.0


In [7]:
df_train_dec.isna().sum()

ClosePrice                     0
AttachedGarageYN            8163
Stories                     7001
ViewYN                      6506
PoolPrivateYN               5268
NewConstructionYN           5151
Levels                      4712
BuyerOfficeAOR              4074
GarageSpaces                2745
LotSizeSquareFeet           1153
LotSizeAcres                1151
LotSizeArea                 1143
BuyerOfficeName              783
ListAgentFirstName           490
BuyerAgentFirstName          297
ListAgentEmail               233
BuyerAgentMlsId               71
StreetNumberNumeric           62
UnparsedAddress               48
FireplaceYN                   47
YearBuilt                     35
BuyerAgentLastName            28
City                          23
ListAgentFullName             19
BuyerAgentAOR                 11
BathroomsTotalInteger         11
ListAgentAOR                  11
ListAgentLastName              8
PostalCode                     1
ParkingTotal                   1
PropertyTy

In [8]:
columns_to_impute = [
    "AttachedGarageYN",
    "Stories",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "Levels",
    "BuyerOfficeAOR",
    "GarageSpaces",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "BuyerOfficeName",
    "ListAgentFirstName",
    "BuyerAgentFirstName",
    "ListAgentEmail",
    "BuyerAgentMlsId",
    "StreetNumberNumeric",
    "UnparsedAddress",
    "FireplaceYN",
    "YearBuilt",
    "BuyerAgentLastName",
    "City",
    "ListAgentFullName",
    "BuyerAgentAOR",
    "BathroomsTotalInteger",
    "ListAgentAOR",
    "ListAgentLastName",
    "PostalCode",
    "ParkingTotal"
]




def random_sample_impute(train_df, test_df, column):
    observed_values = train_df[column].dropna().values
    
    
    train_missing = train_df[column].isna()
    train_df.loc[train_missing, column] = np.random.choice(
        observed_values,
        size=train_missing.sum(),
        replace=True
    )
    
    #
    test_missing = test_df[column].isna()
    test_df.loc[test_missing, column] = np.random.choice(
        observed_values,
        size=test_missing.sum(),
        replace=True
    )
    
    return train_df, test_df

for col in columns_to_impute:
    df_train_dec, df_test_dec = random_sample_impute(
        df_train_dec,
        df_test_dec,
        col
    )
df_train_dec
df_train_dec.isna().sum().sum()
df_test_dec.isna().sum().sum()

np.int64(0)

In [9]:
df_train_jan.isna().sum()

ClosePrice                     0
AttachedGarageYN            8141
Stories                     6689
ViewYN                      6242
PoolPrivateYN               5057
NewConstructionYN           5029
Levels                      4437
BuyerOfficeAOR              4628
GarageSpaces                2671
LotSizeSquareFeet           1113
LotSizeAcres                1112
LotSizeArea                 1106
BuyerOfficeName              734
ListAgentFirstName           470
BuyerAgentFirstName          276
ListAgentEmail               233
BuyerAgentMlsId               82
StreetNumberNumeric           66
UnparsedAddress               48
FireplaceYN                   48
YearBuilt                     32
BuyerAgentLastName            34
City                          13
ListAgentFullName             23
BuyerAgentAOR                 11
BathroomsTotalInteger          7
ListAgentAOR                  11
ListAgentLastName              7
PostalCode                     1
ParkingTotal                   0
PropertyTy

In [10]:
columns_jan = [
    "AttachedGarageYN",
    "Stories",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "Levels",
    "BuyerOfficeAOR",
    "GarageSpaces",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "BuyerOfficeName",
    "ListAgentFirstName",
    "BuyerAgentFirstName",
    "ListAgentEmail",
    "BuyerAgentMlsId",
    "StreetNumberNumeric",
    "UnparsedAddress",
    "FireplaceYN",
    "YearBuilt",
    "BuyerAgentLastName",
    "City",
    "ListAgentFullName",
    "BuyerAgentAOR",
    "BathroomsTotalInteger",
    "ListAgentAOR",
    "ListAgentLastName",
    "PostalCode"
]

for col in columns_jan:
    df_train_jan, df_test_jan = random_sample_impute(
        df_train_jan,
        df_test_jan,
        col
    )




In [11]:
df_train_jan.columns

Index(['ClosePrice', 'AttachedGarageYN', 'Stories', 'ViewYN', 'PoolPrivateYN',
       'NewConstructionYN', 'Levels', 'BuyerOfficeAOR', 'GarageSpaces',
       'LotSizeSquareFeet', 'LotSizeAcres', 'LotSizeArea', 'BuyerOfficeName',
       'ListAgentFirstName', 'BuyerAgentFirstName', 'ListAgentEmail',
       'BuyerAgentMlsId', 'StreetNumberNumeric', 'UnparsedAddress',
       'FireplaceYN', 'YearBuilt', 'BuyerAgentLastName', 'City',
       'ListAgentFullName', 'BuyerAgentAOR', 'BathroomsTotalInteger',
       'ListAgentAOR', 'ListAgentLastName', 'PostalCode', 'ParkingTotal',
       'PropertyType', 'ListingKeyNumeric', 'ListOfficeName', 'CountyOrParish',
       'MlsStatus', 'DaysOnMarket', 'ListingKey', 'LivingArea',
       'PropertySubType', 'ContractStatusChangeDate', 'ListingId',
       'StateOrProvince', 'ListingContractDate', 'Latitude', 'Longitude',
       'BedroomsTotal'],
      dtype='object')

Removed BuyerOfficeAOR, BuyerOfficeName, ListAgentFirstName, BuyerAgentFirstName, ListAgentEmail, BuyerAgentMlsId, StreetNumberNumeric, UnparsedAddress, BuyerAgentLastName, ListAgentFullName, BuyerAgentAOR, ListAgentAOR, ListingKeyNumeric, ListingKey, ListingId, ListAgentLastName, ListOfficeName, ContractStatusChangeDate, and ListingContractDate because they do not seem to have any importance with the close price.

In [ ]:
import statsmodels.formula.api as smf
from sklearn.metrics import r2_score

predictors_clean = [
    "AttachedGarageYN",
    "Stories",
    "ViewYN",
    "PoolPrivateYN",
    "NewConstructionYN",
    "Levels",
    "GarageSpaces",
    "LotSizeSquareFeet",
    "LotSizeAcres",
    "LotSizeArea",
    "FireplaceYN",
    "YearBuilt",
    "City",
    "BathroomsTotalInteger",
    "PostalCode",
    "ParkingTotal",
    "PropertyType",
    "CountyOrParish",
    "MlsStatus",
    "DaysOnMarket",
    "LivingArea",
    "PropertySubType",
    "StateOrProvince",
    "Latitude",
    "Longitude",
    "BedroomsTotal"
]

categorical_columns = [
    col for col in predictors_clean
    if df_train_dec[col].dtype == "object"
]

for col in categorical_columns:
    # work with plain objects/strings first
    df_train_dec[col] = df_train_dec[col].astype("object")
    df_test_dec[col]  = df_test_dec[col].astype("object")

    # categories seen in training
    train_cats = set(df_train_dec[col].dropna().unique())

    # map unseen test categories -> "Other"
    df_test_dec.loc[~df_test_dec[col].isin(train_cats) & df_test_dec[col].notna(), col] = "Other"

    # ensure "Other" exists in training (so it becomes a valid category)
    if "Other" not in train_cats:
        df_train_dec.loc[df_train_dec[col].isna(), col] = "Other"  # minimal way to introduce it

    # now convert to category and align
    df_train_dec[col] = df_train_dec[col].astype("category")
    df_test_dec[col]  = df_test_dec[col].astype("category").cat.set_categories(df_train_dec[col].cat.categories)
formula = "ClosePrice ~ " + " + ".join([f'C({c})' if c in categorical_columns else c for c in predictors_clean])

model = smf.ols(formula=formula, data=df_train_dec).fit()

print(model.summary())



actual = df_test_dec["ClosePrice"]
pred_vals = model.predict(df_test_dec)

# actual = df_test_cleaned["ClosePrice"]

mask = actual.notna() & pd.Series(pred_vals).notna()
r_squared_lin = r2_score(actual[mask], pd.Series(pred_vals)[mask])


mape_lin = mape(actual[mask], pd.Series(pred_vals)[mask])
print("Rsquared linear: ", r_squared_lin)
print("MaPe linear: ", mape_lin)

                            OLS Regression Results                            
Dep. Variable:             ClosePrice   R-squared:                       0.169
Model:                            OLS   Adj. R-squared:                  0.144
Method:                 Least Squares   F-statistic:                     6.760
Date:                Sat, 28 Feb 2026   Prob (F-statistic):               0.00
Time:                        00:42:03   Log-Likelihood:            -1.1843e+06
No. Observations:               68286   AIC:                         2.373e+06
Df Residuals:                   66285   BIC:                         2.391e+06
Df Model:                        2000                                         
Covariance Type:            nonrobust                                         
                                           coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------


,ClosePrice,AttachedGarageYN,Stories,ViewYN,PoolPrivateYN,NewConstructionYN,Levels,BuyerOfficeAOR,GarageSpaces,LotSizeSquareFeet,...,ListingKey,LivingArea,PropertySubType,ContractStatusChangeDate,ListingId,StateOrProvince,ListingContractDate,Latitude,Longitude,BedroomsTotal


In [ ]:
r_squared

NameError: name 'r_squared' is not defined